# Final Project - Chem 277B
### **Structure-Aware Resistance Prediction in Mycobacterium tuberculosis via MIC Regression**

### Contributors:
Cris Zong, Ethan Chan, Isabella Beatrice Bonomi, Robert Craig Wallace, Sidney Alexa Brooks

### 1) Objective and Goal of the Project

Objective: To develop a machine learning model that predicts M. tuberculosis drug resistance by jointly encoding mutation profiles and drug molecular structure, rather than treating each drug as an independent categorical label.

Goal: To accurately predict resistance confidence levels from mutation loci and Morgan fingerprints, and ultimately generalize to novel anti-TB compounds not present in existing catalogues.

**Note:** prior to going through this walkthrough, instructions for downloading data will be included in the README markdown file.

In [1]:
# Import standard libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import warnings                                                   #sometimes GMM produces warnings which are not relevant
from sklearn.mixture import GaussianMixture                       #for performing actual GMM
from sklearn.metrics import silhouette_samples, silhouette_score  #calculating silhouette coefficient
from sklearn import datasets                                      #we want to work with an internal data set


In [16]:
# Load training data
master_file = pd.read_csv("WHO-UCN-TB-2023.6-eng_catalogue_master_file.txt", sep=" ")

# Let's take a look at the data
master_file.head()

/var/folders/hp/mt4w7y_15d3fx0bx87yjr2mh0000gn/T/ipykernel_35851/3450041261.py:2: DtypeWarning: Columns (28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57) have mixed types. Specify dtype option on import or set low_memory=False.
  master_file = pd.read_csv("WHO-UCN-TB-2023.6-eng_catalogue_master_file.txt", sep=" ")


,drug\tgene\tmutation\tvariant\ttier\teffect\tgenomic,position\talgorithm_pass_DATASET,ALL\tPresent_SOLO_SR_DATASET,ALL\tPresent_SOLO_R_DATASET,ALL\tPresent_SOLO_S_DATASET,ALL\tPresent_R_DATASET,ALL\tPresent_S_DATASET,ALL\tAbsent_R_DATASET,ALL\tAbsent_S_DATASET,ALL\tSens_DATASET,...,"CFZ_Rv0678,","INH_katG,",DLM_ddn/fbiA/fbiB/fbiC/fgd1/Rv2983)\tSilent,mutation\tListed,in,abridged,tables\tAdditional,grading\tFootnote\tCHANGES,vs.1,ver1
0,Amikacin\tbacA\tc.102G>A\tbacA_c.102G>A\t2\tsy...,Genomic_coordinates,sheet)\tNA\tNA\tNA\tNA\t0\t1\t2460\t21977\t0\t...,Uncertain,significance\t3),Uncertain,significance\tALL+WHO\t3),Uncertain,significance\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\t...,mutation\t4),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amikacin\tbacA\tc.1044G>A\tbacA_c.1044G>A\t2\t...,Genomic_coordinates,sheet)\tNA\tNA\tNA\tNA\t91\t1112\t2369\t20866\...,Not,assoc,w,R\t5),Not,assoc,w,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Amikacin\tbacA\tc.105C>G\tbacA_c.105C>G\t2\tsy...,Genomic_coordinates,sheet)\tNA\tNA\tNA\tNA\t0\t1\t2460\t21977\t0\t...,Uncertain,significance\t3),Uncertain,significance\tALL+WHO\t3),Uncertain,significance\tNA\tNA\tNA\tNA\t0\t1\t1111\t7772...,mutation\t4),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amikacin\tbacA\tc.1065T>G\tbacA_c.1065T>G\t2\t...,Genomic_coordinates,sheet)\tNA\tNA\tNA\tNA\t0\t3\t2460\t21975\t0\t...,Uncertain,significance\t3),Uncertain,significance\tALL+WHO\t3),Uncertain,significance\tNA\tNA\tNA\tNA\t0\t2\t1111\t7771...,mutation\t4),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Amikacin\tbacA\tc.1080G>A\tbacA_c.1080G>A\t2\t...,Genomic_coordinates,sheet)\tNA\tNA\tNA\tNA\t0\t2\t2460\t21976\t0\t...,Uncertain,significance\t3),Uncertain,significance\tALL+WHO\t3),Uncertain,significance\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\t...,mutation\t4),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
variants_excel = pd.read_excel("WHO-UCN-TB-2023.7-eng.xlsx")

# Let's take a look at the data
variants_excel.head()

/Users/Bella/miniconda3/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DATASET ALL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,drug,gene,mutation,variant,tier,effect,genomic position,algorithm_pass,Present_SOLO_SR,Present_SOLO_R,...,Additional grading criteria applied,FINAL CONFIDENCE GRADING,Comment,CHANGES vs ver1,"Relaxed thresholds simulation (BDQ_Rv0678, CFZ...",Silent mutation,Listed in abridged tables,Additional grading,Footnote,CHANGES vs ver1
2,Amikacin,bacA,c.102G>A,bacA_c.102G>A,2,synonymous_variant,"(see ""Genomic_coordinates"" sheet)",NaN,NaN,NaN,...,Silent mutation,4) Not assoc w R - Interim,NaN,Now listed,NaN,Silent mutation,no,NaN,NaN,0
3,Amikacin,bacA,c.1044G>A,bacA_c.1044G>A,2,synonymous_variant,"(see ""Genomic_coordinates"" sheet)",NaN,NaN,NaN,...,NaN,5) Not assoc w R,NaN,Now listed,NaN,Silent mutation,no,NaN,NaN,0
4,Amikacin,bacA,c.105C>G,bacA_c.105C>G,2,synonymous_variant,"(see ""Genomic_coordinates"" sheet)",NaN,NaN,NaN,...,Silent mutation,4) Not assoc w R - Interim,NaN,Now listed,NaN,Silent mutation,no,NaN,NaN,0


In [14]:
# Load genomic data
genomic_data = pd.read_csv("WHO-UCN-TB-2023.7-eng_genomic_coordinates.txt", sep="\t")

# Let's take a look at the data
genomic_data.head()


,variant,chromosome,position,reference_nucleotide,alternative_nucleotide
0,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CA
1,dnaA_p.Asp3Ala,NC_000962.3,8,A,C
2,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CC
3,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CG
4,dnaA_p.Asp4His,NC_000962.3,10,G,C
